# 02 Data Cleaning and Relational Migration

Day 2 notebook for sanitizing the raw Bluestock datasets and loading the cleaned star schema into SQLite.

## Cleaning Rules

1. Normalize text fields and preserve canonical AMFI scheme identifiers.
2. Resample NAV history to a continuous daily frequency and forward-fill missing market days.
3. Filter invalid monetary records so `amount_inr > 0` and `nav > 0`.
4. Standardize transaction types into the capitalized flags `SIP`, `Lumpsum`, and `Redemption`.
5. Keep only the columns required by the star schema before migration.

## Star Schema Design Choices

- `dim_fund` stores the static mutual fund dimension.
- `fact_nav` stores one daily NAV observation per scheme.
- `fact_transactions` stores investor-level transaction activity.
- SQLite is used as the local relational warehouse for the capstone workflow.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)

RAW_DIR = Path('../data/raw')
if not RAW_DIR.exists():
    RAW_DIR = Path('data/raw')

PROCESSED_DIR = Path('../data/processed')
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = Path('data/processed')

DB_PATH = Path('../data/db/bluestock_mf.db')
if not DB_PATH.parent.exists():
    DB_PATH = Path('data/db/bluestock_mf.db')

SCHEMA_PATH = Path('../sql/schema.sql')
if not SCHEMA_PATH.exists():
    SCHEMA_PATH = Path('sql/schema.sql')

SOURCE_FILES = {
    'fund_master': '01_fund_master.csv',
    'nav_history': '02_nav_history.csv',
    'aum_by_fund_house': '03_aum_by_fund_house.csv',
    'monthly_sip_inflows': '04_monthly_sip_inflows.csv',
    'category_inflows': '05_category_inflows.csv',
    'industry_folio_count': '06_industry_folio_count.csv',
    'scheme_performance': '07_scheme_performance.csv',
    'investor_transactions': '08_investor_transactions.csv',
    'portfolio_holdings': '09_portfolio_holdings.csv',
    'benchmark_indices': '10_benchmark_indices.csv',
}

raw_datasets = {
    label: pd.read_csv(RAW_DIR / filename)
    for label, filename in SOURCE_FILES.items()
}

print('Raw dataset inventory loaded from ../data/raw/')
for label, df in raw_datasets.items():
    print(f'{label}: shape={df.shape}')

Raw dataset inventory loaded from ../data/raw/
fund_master: shape=(40, 15)
nav_history: shape=(46000, 3)
aum_by_fund_house: shape=(90, 5)
monthly_sip_inflows: shape=(48, 6)
category_inflows: shape=(144, 3)
industry_folio_count: shape=(21, 6)
scheme_performance: shape=(40, 19)
investor_transactions: shape=(32778, 13)
portfolio_holdings: shape=(322, 8)
benchmark_indices: shape=(8050, 3)


In [2]:
def clean_fund_master(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()
    clean.columns = [c.strip() for c in clean.columns]
    clean['amfi_code'] = pd.to_numeric(clean['amfi_code'], errors='coerce')
    clean['expense_ratio_pct'] = pd.to_numeric(clean['expense_ratio_pct'], errors='coerce')
    clean['exit_load_pct'] = pd.to_numeric(clean['exit_load_pct'], errors='coerce')
    clean['min_sip_amount'] = pd.to_numeric(clean['min_sip_amount'], errors='coerce')
    clean['min_lumpsum_amount'] = pd.to_numeric(clean['min_lumpsum_amount'], errors='coerce')
    clean['launch_date'] = pd.to_datetime(clean['launch_date'], errors='coerce')

    text_cols = [
        'fund_house', 'scheme_name', 'category', 'sub_category',
        'plan', 'benchmark', 'fund_manager', 'risk_category', 'sebi_category_code'
    ]
    for col in text_cols:
        clean[col] = clean[col].astype(str).str.strip()

    clean['risk_category'] = clean['risk_category'].str.title()
    clean['category'] = clean['category'].str.title()
    clean['sub_category'] = clean['sub_category'].str.title()
    clean['plan'] = clean['plan'].str.title()

    clean = clean.dropna(subset=[
        'amfi_code', 'fund_house', 'scheme_name', 'category',
        'sub_category', 'expense_ratio_pct', 'risk_category'
    ])
    clean['amfi_code'] = clean['amfi_code'].astype(int)
    clean = clean.drop_duplicates(subset=['amfi_code']).sort_values('amfi_code').reset_index(drop=True)

    return clean[[
        'amfi_code', 'fund_house', 'scheme_name', 'category',
        'sub_category', 'expense_ratio_pct', 'risk_category'
    ]]


def clean_nav_history(df: pd.DataFrame) -> pd.DataFrame:
    nav = df.copy()
    nav['amfi_code'] = pd.to_numeric(nav['amfi_code'], errors='coerce')
    nav['nav_date'] = pd.to_datetime(nav['date'], errors='coerce')
    nav['nav'] = pd.to_numeric(nav['nav'], errors='coerce')
    nav = nav.dropna(subset=['amfi_code', 'nav_date', 'nav'])
    nav['amfi_code'] = nav['amfi_code'].astype(int)
    nav = nav.sort_values(['amfi_code', 'nav_date'])

    frames = []
    for amfi_code, group in nav.groupby('amfi_code', sort=True):
        daily = (
            group.set_index('nav_date')[['nav']]
            .sort_index()
            .resample('D')
            .ffill()
            .reset_index()
        )
        daily['amfi_code'] = int(amfi_code)
        frames.append(daily[['amfi_code', 'nav_date', 'nav']])

    clean = pd.concat(frames, ignore_index=True)
    clean = clean[clean['nav'] > 0].drop_duplicates(subset=['amfi_code', 'nav_date'], keep='last')
    clean['nav_date'] = pd.to_datetime(clean['nav_date']).dt.strftime('%Y-%m-%d')
    return clean.sort_values(['amfi_code', 'nav_date']).reset_index(drop=True)


def clean_transactions(df: pd.DataFrame) -> pd.DataFrame:
    tx = df.copy()
    tx['investor_id'] = tx['investor_id'].astype(str).str.strip()
    tx['amfi_code'] = pd.to_numeric(tx['amfi_code'], errors='coerce')
    tx['transaction_date'] = pd.to_datetime(tx['transaction_date'], errors='coerce')
    tx['amount_inr'] = pd.to_numeric(tx['amount_inr'], errors='coerce')
    tx['transaction_type'] = tx['transaction_type'].astype(str).str.strip().str.lower()
    tx['state'] = tx['state'].astype(str).str.strip().str.title()
    tx['city_tier'] = tx['city_tier'].astype(str).str.strip().str.upper()
    tx['age_group'] = tx['age_group'].astype(str).str.strip()

    type_map = {
        'sip': 'SIP',
        'lumpsum': 'Lumpsum',
        'redemption': 'Redemption',
    }
    tx['transaction_type'] = tx['transaction_type'].map(type_map).fillna(tx['transaction_type'].str.title())

    invalid_amounts = int((tx['amount_inr'] <= 0).sum())
    print(f'Invalid amount boundary count before filtering: {invalid_amounts}')

    tx = tx.dropna(subset=['investor_id', 'amfi_code', 'transaction_date', 'amount_inr'])
    tx = tx[tx['amount_inr'] > 0].copy()
    tx['amfi_code'] = tx['amfi_code'].astype(int)
    tx['amount_inr'] = tx['amount_inr'].astype(int)
    tx['transaction_date'] = tx['transaction_date'].dt.strftime('%Y-%m-%d')
    tx = tx.drop_duplicates().reset_index(drop=True)
    return tx[[
        'investor_id', 'amfi_code', 'transaction_date', 'amount_inr',
        'transaction_type', 'state', 'city_tier'
    ]]


def clean_scheme_performance(df: pd.DataFrame) -> pd.DataFrame:
    perf = df.copy()
    perf.columns = [c.strip() for c in perf.columns]
    perf['amfi_code'] = pd.to_numeric(perf['amfi_code'], errors='coerce')
    numeric_cols = [
        'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct',
        'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct',
        'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating'
    ]
    for col in numeric_cols:
        perf[col] = pd.to_numeric(perf[col], errors='coerce')

    text_cols = ['scheme_name', 'fund_house', 'category', 'plan', 'risk_grade']
    for col in text_cols:
        perf[col] = perf[col].astype(str).str.strip()

    perf['category'] = perf['category'].str.title()
    perf['plan'] = perf['plan'].str.title()
    perf['risk_grade'] = perf['risk_grade'].str.title()

    perf = perf.dropna(subset=['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan'])
    perf['amfi_code'] = perf['amfi_code'].astype(int)
    perf['morningstar_rating'] = perf['morningstar_rating'].round().astype('Int64')
    perf = perf.drop_duplicates(subset=['amfi_code']).sort_values('amfi_code').reset_index(drop=True)

    return perf[[
        'amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
        'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct',
        'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct',
        'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating',
        'risk_grade'
    ]]


fund_master_clean = clean_fund_master(raw_datasets['fund_master'])
nav_history_clean = clean_nav_history(raw_datasets['nav_history'])
transactions_clean = clean_transactions(raw_datasets['investor_transactions'])
scheme_performance_clean = clean_scheme_performance(raw_datasets['scheme_performance'])

print()
print('Cleaned output shapes')
print(f'dim_fund: {fund_master_clean.shape}')
print(f'fact_nav: {nav_history_clean.shape}')
print(f'fact_transactions: {transactions_clean.shape}')
print(f'clean_scheme_performance: {scheme_performance_clean.shape}')

Invalid amount boundary count before filtering: 0

Cleaned output shapes
dim_fund: (40, 7)
fact_nav: (64320, 3)
fact_transactions: (32778, 7)
clean_scheme_performance: (40, 19)


In [3]:
# Persist cleaned CSVs for downstream notebook reuse.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

fund_master_clean.to_csv(PROCESSED_DIR / 'clean_01_fund_master.csv', index=False)
nav_history_clean.to_csv(PROCESSED_DIR / 'clean_02_nav_history.csv', index=False)
transactions_clean.to_csv(PROCESSED_DIR / 'clean_08_investor_transactions.csv', index=False)
scheme_performance_clean.to_csv(PROCESSED_DIR / 'clean_07_scheme_performance.csv', index=False)

print('Cleaned CSV exports written to ../data/processed/')

Cleaned CSV exports written to ../data/processed/


In [4]:
# Load the cleaned star schema into SQLite using SQLAlchemy.
engine = create_engine(f"sqlite:///{DB_PATH.resolve().as_posix()}", future=True)

schema_sql = SCHEMA_PATH.read_text(encoding='utf-8')
statements = [stmt.strip() for stmt in schema_sql.split(';') if stmt.strip()]

with engine.begin() as connection:
    connection.exec_driver_sql('PRAGMA foreign_keys = ON')
    connection.exec_driver_sql('DROP TABLE IF EXISTS fact_transactions')
    connection.exec_driver_sql('DROP TABLE IF EXISTS fact_nav')
    connection.exec_driver_sql('DROP TABLE IF EXISTS dim_fund')

    for statement in statements:
        connection.exec_driver_sql(statement)

fund_master_clean.to_sql('dim_fund', con=engine, if_exists='append', index=False)
nav_history_clean.to_sql('fact_nav', con=engine, if_exists='append', index=False)
transactions_clean.to_sql('fact_transactions', con=engine, if_exists='append', index=False)

print('SQLite migration completed successfully.')

SQLite migration completed successfully.


In [5]:
# Verification snapshot after migration.
with engine.connect() as connection:
    dim_count = connection.exec_driver_sql('SELECT COUNT(*) FROM dim_fund').scalar_one()
    nav_count = connection.exec_driver_sql('SELECT COUNT(*) FROM fact_nav').scalar_one()
    tx_count = connection.exec_driver_sql('SELECT COUNT(*) FROM fact_transactions').scalar_one()

print(f'dim_fund rows: {dim_count}')
print(f'fact_nav rows: {nav_count}')
print(f'fact_transactions rows: {tx_count}')

dim_fund rows: 40
fact_nav rows: 64320
fact_transactions rows: 32778


## Migration Completion

The cleaned star schema is now loaded into SQLite and the processed CSV layers are ready for downstream analysis notebooks.